<a href="https://colab.research.google.com/github/santiago-zuluaga/Algoritmo-Genetico/blob/main/notebooks/algoritmo_genetico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/></a>

# Laboratorio: algoritmo genético y estrella mágica

En este cuaderno construiremos la idea paso a paso: definiremos el problema, generaremos una población aleatoria, observaremos el fitness y los operadores, y al final seguiremos la evolución generación por generación. El algoritmo y sus operadores se importan del proyecto; aquí nos enfocamos en observarlos.
La secuencia sigue el formato de la clase práctica: fijar parámetros, generar población, definir y comprobar la evaluación, estudiar cada operador y finalmente ejecutar el algoritmo completo.


## 1. ¿Qué es un algoritmo genético?

Es una búsqueda inspirada en la evolución. Tenemos varias soluciones candidatas, medimos qué tan buenas son, elegimos algunas para reproducirse y generamos nuevas candidatas mediante cruce y mutación. Repetimos el proceso en generaciones. En este parcial se minimiza el error de las seis líneas de la estrella. No se garantiza encontrar una solución en toda corrida.
| Término | En este problema |
|---|---|
| **Población** | Conjunto de distribuciones candidatas de los números 1–12. |
| **Individuo** | Una estrella candidata completa. |
| **Cromosoma** | La lista ordenada de 12 genes que representa al individuo. |
| **Gen** | Un número en una posición A–L. |
| **Fitness** | Suma de los errores absolutos de las seis líneas; menor es mejor. |
| **Generación** | Una iteración que reemplaza la población por descendientes y élites. |
| **Selección** | Escoge padres; el torneo favorece fitness bajo. |
| **Cruce** | Combina dos padres; OX mantiene el orden y evita duplicados. |
| **Mutación** | Intercambia dos posiciones para explorar otra distribución. |
| **Elitismo** | Copia los mejores individuos sin cambiarlos a la generación siguiente. |

## 2. Preparar Google Colab y cargar el proyecto

Esta celda clona el repositorio en el directorio temporal de Colab (si todavía no existe), instala las dependencias gráficas y carga los módulos Python del proyecto. No utiliza rutas de Windows.

In [ ]:
from pathlib import Path
import os, subprocess, sys, random, time
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display

# Preparación opcional de Google Colab; las operaciones genéticas se implementan aquí.
if "google.colab" in sys.modules:
    REPO = Path('/content/Algoritmo-Genetico')
    if not REPO.exists():
        subprocess.run(['git','clone','https://github.com/santiago-zuluaga/Algoritmo-Genetico.git',str(REPO)],check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable,'-m','pip','install','-q','matplotlib','ipywidgets','pandas','numpy'],check=True)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

NUMEROS=tuple(range(1,13)); ETIQUETAS=tuple('ABCDEFGHIJKL')
LINEAS=((0,2,5,7),(0,3,6,10),(1,2,3,4),(1,5,8,11),(4,6,9,11),(7,8,9,10))
def validar_individuo(ind): return len(ind)==12 and sorted(ind)==list(NUMEROS)
def sumas_lineas(ind):
    if not validar_individuo(ind): raise ValueError('Se requiere una permutación de 1..12')
    return tuple(sum(ind[i] for i in linea) for linea in LINEAS)
def fitness(ind): return sum(abs(s-26) for s in sumas_lineas(ind))
def crear_individuo(rng):
    ind=list(NUMEROS); rng.shuffle(ind); return ind
def seleccion_torneo(poblacion,tamano,rng):
    if not 1<=tamano<=len(poblacion): raise ValueError('Tamaño de torneo inválido')
    return min(rng.sample(poblacion,tamano),key=fitness)
def cruce_ox_detallado(a,b,rng):
    if not validar_individuo(a) or not validar_individuo(b): raise ValueError('Padres inválidos')
    i,j=sorted(rng.sample(range(12),2))
    def hijo(x,y):
        h=[None]*12; h[i:j+1]=x[i:j+1]
        resto=[g for g in y[j+1:]+y[:j+1] if g not in h]
        posiciones=list(range(j+1,12))+list(range(i))
        for pos,g in zip(posiciones,resto): h[pos]=g
        return h
    h1=hijo(a,b); h2=hijo(b,a)
    resto=[g for g in b[j+1:]+b[:j+1] if g not in a[i:j+1]]
    posiciones=list(range(j+1,12))+list(range(i))
    return (h1,h2),(i,j),{'segmento':a[i:j+1],'relleno':resto,'pasos':list(zip(posiciones,resto))}
def mutacion_swap_detallada(ind,rng,probabilidad=1.0):
    h=ind.copy(); posiciones=None
    if rng.random()<probabilidad:
        i,j=rng.sample(range(12),2); h[i],h[j]=h[j],h[i]; posiciones=(i,j)
    return h,posiciones
@dataclass
class ConfiguracionAG:
    tamano_poblacion:int=100
    max_generaciones:int=300
    probabilidad_mutacion:float=.15
    elitismo:int=2
    tamano_torneo:int=3
    semilla:int=2026
def avanzar_poblacion(poblacion,config,rng):
    nueva=[x.copy() for x in sorted(poblacion,key=fitness)[:config.elitismo]]
    while len(nueva)<len(poblacion):
        padres=[seleccion_torneo(poblacion,config.tamano_torneo,rng) for _ in range(2)]
        hijos,_,_=cruce_ox_detallado(*padres,rng)
        for h in hijos:
            nueva.append(mutacion_swap_detallada(h,rng,config.probabilidad_mutacion)[0])
            if len(nueva)==len(poblacion): break
    return nueva
def crear_instantanea(generacion,poblacion):
    valores=[fitness(x) for x in poblacion]; k=int(np.argmin(valores))
    return {'generacion':generacion,'poblacion':[x.copy() for x in poblacion],
            'fitnesses':valores,'mejor_individuo':poblacion[k].copy(),
            'mejor_fitness':valores[k],'fitness_promedio':float(np.mean(valores)),
            'peor_fitness':max(valores),'diversidad':len({tuple(x) for x in poblacion})/len(poblacion)}
def ejecutar_con_historial(config):
    rng=random.Random(config.semilla)
    poblacion=[crear_individuo(rng) for _ in range(config.tamano_poblacion)]
    historial=[crear_instantanea(0,poblacion)]
    for g in range(1,config.max_generaciones+1):
        if historial[-1]['mejor_fitness']==0: break
        poblacion=avanzar_poblacion(poblacion,config,rng)
        historial.append(crear_instantanea(g,poblacion))
    return historial
print('Implementación genética cargada en esta celda; Colab:', 'google.colab' in sys.modules)

## 3. Representación del cromosoma

Cada cromosoma es una permutación `[A, B, ..., L]` de 1 a 12. Las líneas son `A+C+F+H`, `A+D+G+K`, `B+C+D+E`, `B+F+I+L`, `E+G+J+L` y `H+I+J+K`. El objetivo es que cada suma sea 26. La lógica de validación y fitness está visible arriba.

In [ ]:
muestra=list(range(1,13))
print('Posiciones:', dict(zip(ETIQUETAS,muestra)))
print('Permutación válida:', validar_individuo(muestra))
print('Sumas de líneas:', sumas_lineas(muestra), '| fitness:', fitness(muestra))
assert validar_individuo(muestra) and len(sumas_lineas(muestra))==6

## 4. Generar la población inicial

Como en el ejemplo de clase, primero fijamos los parámetros y luego generamos la población. Cada individuo se crea mezclando aleatoriamente los números 1–12; no insertamos la solución de referencia. La semilla permite repetir la misma demostración.

In [ ]:
#@title Parámetros de la población para el ejemplo
tamano_demo = 100 #@param {type:"integer"}
semilla_demo = 2026 #@param {type:"integer"}
tamano_demo = max(4, int(tamano_demo))
rng_demo = random.Random(semilla_demo)
poblacion = [crear_individuo(rng_demo) for _ in range(tamano_demo)]
assert all(validar_individuo(ind) for ind in poblacion)

def tabla_poblacion(poblacion, cantidad=5):
    ordenados = sorted(poblacion, key=fitness)[:cantidad]
    return pd.DataFrame([{
        'Individuo': i+1, 'Cromosoma A–L': str(ind), 'Fitness': fitness(ind),
        'Seis sumas': str(sumas_lineas(ind))
    } for i, ind in enumerate(ordenados)])

fitnesses = [fitness(ind) for ind in poblacion]
display(tabla_poblacion(poblacion, 5))
print(f'Población: {len(poblacion)} individuos | mejor fitness: {min(fitnesses)} | promedio: {sum(fitnesses)/len(fitnesses):.2f}')
fig, ax = plt.subplots(figsize=(7, 5)); ax.hist(fitnesses, bins=15, color='#4776a8', edgecolor='white')
ax.set(title='Fitness de la población inicial', xlabel='Fitness (menor es mejor)', ylabel='Cantidad de individuos'); ax.grid(alpha=.2); plt.show(); plt.close(fig)

La tabla muestra cinco individuos con menor fitness, y el histograma deja ver la distribución completa. Que un individuo sea el mejor de esta población no significa que sea una solución: comprobamos las seis sumas, y sólo fitness cero cumple el problema.

## 5. Selección por torneo

El torneo toma una muestra de candidatos y elige el de menor fitness. Aquí se muestran los participantes antes de seleccionar al ganador. Si dos empatan, cualquiera de los empatados puede ganar según el orden del sorteo.

In [ ]:
rng_torneo = random.Random(17)
candidatos = rng_torneo.sample(poblacion, 3)
ganador = min(candidatos, key=fitness)
tabla_torneo = pd.DataFrame([{
    'Cromosoma': str(ind), 'Fitness': fitness(ind),
    'Resultado': 'GANADOR' if ind is ganador else 'participante'
} for ind in candidatos])
display(tabla_torneo)
print('Se selecciona el menor fitness entre estos tres candidatos.')
assert seleccion_torneo(poblacion, 3, random.Random(17)) is ganador

## 6. Cruce Order Crossover (OX)

OX copia un segmento del primer padre en las mismas posiciones del hijo. Luego recorre el segundo padre en orden circular, salta los genes ya copiados y completa los espacios libres. Esto es apropiado para permutaciones porque evita números repetidos y conserva todos los genes.

In [ ]:
padre_1, padre_2 = poblacion[0], poblacion[1]
(hijo_1, hijo_2), (inicio, fin), detalle_ox = cruce_ox_detallado(padre_1, padre_2, random.Random(31))
tabla_ox = pd.DataFrame({
    'Índice': list(range(12)), 'Padre 1': padre_1, 'Padre 2': padre_2,
    'Hijo 1': hijo_1,
    'Parte heredada': ['sí' if inicio <= i <= fin else '' for i in range(12)]
})
display(tabla_ox)
print(f'Segmento heredado de Padre 1: índices {inicio}–{fin} → {detalle_ox["segmento"]}')
print('Orden de relleno desde Padre 2:', detalle_ox['relleno'])
print('Asignaciones (índice, gen):', detalle_ox['pasos'])
print('Hijo válido:', validar_individuo(hijo_1), '| genes únicos:', len(set(hijo_1)))
assert validar_individuo(hijo_1) and validar_individuo(hijo_2)

## 7. Mutación por intercambio

La mutación swap escoge dos posiciones y cambia sus valores. No agrega ni elimina números: el resultado sigue siendo una permutación. Una mutación ocasional ayuda a visitar distribuciones que el cruce de padres actuales podría no producir.

In [ ]:
antes_mutacion = hijo_1.copy()
despues_mutacion, posiciones_mutadas = mutacion_swap_detallada(antes_mutacion, random.Random(8), probabilidad=1.0)
print('Antes:  ', antes_mutacion)
print('Después:', despues_mutacion)
print('Posiciones intercambiadas:', posiciones_mutadas)
print('Fitness antes/después:', fitness(antes_mutacion), fitness(despues_mutacion))
assert validar_individuo(despues_mutacion)

## 8. Elitismo y construcción de una generación

El elitismo copia sin cambios a los mejores individuos. El resto de la generación se completa con padres seleccionados por torneo, cruce OX y mutación swap. El módulo del proyecto realiza esta secuencia en `avanzar_poblacion`.

In [ ]:
config_demo = ConfiguracionAG(tamano_poblacion=100, max_generaciones=100, probabilidad_mutacion=.15, elitismo=2, tamano_torneo=3, semilla=2026)
rng_paso = random.Random(config_demo.semilla)
siguiente = avanzar_poblacion(poblacion, config_demo, rng_paso)
elite_esperada = sorted(poblacion, key=fitness)[:config_demo.elitismo]
print('Élite que se copia:')
display(tabla_poblacion(elite_esperada, config_demo.elitismo))
print('¿Los élites permanecen en la nueva población?', all(ind in siguiente for ind in elite_esperada))
print('Tamaño de la nueva población:', len(siguiente))
assert all(ind in siguiente for ind in elite_esperada)

## 12. Preguntas para comprobar lo aprendido

1. ¿Por qué cada cromosoma debe ser una permutación de 1 a 12?
2. ¿Qué ventaja tiene OX frente a un cruce que copie genes sin revisar duplicados?
3. Si una línea suma 24, ¿cuánto aporta al fitness?
4. ¿Qué puede pasar con la búsqueda si la diversidad llega casi a cero?
5. ¿Cómo puede ayudar la mutación y qué ocurre si es excesiva?
6. ¿Por qué el elitismo copia algunos individuos sin modificarlos?
7. ¿Por qué el algoritmo no garantiza encontrar fitness cero en un número fijo de generaciones?

## Conclusión

Un algoritmo genético combina selección, cruce, mutación y elitismo para buscar mejores permutaciones. Las estadísticas y la estrella animada permiten observar tanto el avance como el estancamiento; el fitness cero es la comprobación de éxito, no el mero hecho de terminar una corrida.

## Visualización de la población inicial: fitness y cromosomas

El histograma resume la población entera. El mapa de calor usa filas para individuos, columnas A–L y colores para los genes.

In [ ]:
def graficar_poblacion(ax_hist,ax_heat,poblacion,titulo):
    vals=[fitness(x) for x in poblacion]
    ax_hist.clear(); ax_heat.clear()
    ax_hist.hist(vals,bins=np.arange(min(vals)-.5,max(vals)+1.5,1),color='#4776a8',edgecolor='white')
    ax_hist.set(title=titulo+' · fitness',xlabel='Fitness',ylabel='Individuos')
    im=ax_heat.imshow(np.asarray(poblacion),aspect='auto',interpolation='nearest',cmap='viridis',vmin=1,vmax=12)
    ax_heat.set(title=titulo+' · población completa',xlabel='Posición',ylabel='Individuo'); ax_heat.set_xticks(range(12),ETIQUETAS)
    return im
fig,(a,b)=plt.subplots(1,2,figsize=(14,5)); im=graficar_poblacion(a,b,poblacion,'Generación 0')
fig.colorbar(im,ax=b,label='Valor del gen'); plt.tight_layout(); plt.show()

El histograma y el mapa muestran tendencias agregadas y la población completa, sin dibujar estrellas por individuo.

## Evolución completa con instantáneas

Cada instantánea guarda población y fitness completos, mejor cromosoma, promedio, peor valor y diversidad. La corrida se puede reconstruir sin recalcular operadores.

In [ ]:
tamano_corrida=100 #@param {type:'integer'}
generaciones_corrida=300 #@param {type:'integer'}
mutacion_corrida=0.15 #@param {type:'number'}
elitismo_corrida=2 #@param {type:'integer'}
torneo_corrida=3 #@param {type:'integer'}
semilla_corrida=2026 #@param {type:'integer'}
config=ConfiguracionAG(tamano_poblacion=int(tamano_corrida),max_generaciones=int(generaciones_corrida),probabilidad_mutacion=float(mutacion_corrida),elitismo=int(elitismo_corrida),tamano_torneo=int(torneo_corrida),semilla=int(semilla_corrida))
t0=time.perf_counter(); historial=ejecutar_con_historial(config); segundos=time.perf_counter()-t0
print('Generaciones:',len(historial)-1,'tiempo:',round(segundos,2),'s','fitness final:',historial[-1]['mejor_fitness'])
assert all(len(x['poblacion'])==config.tamano_poblacion for x in historial)
assert all(all(validar_individuo(i) for i in x['poblacion']) for x in historial)
display(pd.DataFrame([{k:x[k] for k in ('generacion','mejor_fitness','fitness_promedio','peor_fitness','diversidad')} for x in historial]).tail())

## Comparación de generaciones

El control selecciona una generación intermedia. Cada panel incluye histograma, mapa de calor y estadísticas de generación 0, la intermedia y la última.

In [ ]:
def comparar_generaciones(intermedia):
    indices=[0,int(intermedia),len(historial)-1]
    fig,axs=plt.subplots(3,2,figsize=(14,12),gridspec_kw={'width_ratios':[1,1.5]})
    for fila,idx in enumerate(indices):
        s=historial[idx]; im=graficar_poblacion(axs[fila,0],axs[fila,1],s['poblacion'],f"Generación {s['generacion']}")
        axs[fila,0].text(.98,.95,f"mejor {s['mejor_fitness']} · media {s['fitness_promedio']:.1f} · diversidad {s['diversidad']:.2f}",transform=axs[fila,0].transAxes,ha='right',va='top',bbox={'facecolor':'white','alpha':.8})
    fig.colorbar(im,ax=axs[:,1].tolist(),label='Valor del gen'); plt.tight_layout(); plt.show()
if len(historial)>2:
    control_intermedia=widgets.IntSlider(value=len(historial)//2,min=1,max=len(historial)-2,description='Intermedia')
    display(widgets.interactive_output(comparar_generaciones,{'intermedia':control_intermedia}),control_intermedia)
else: comparar_generaciones(0)

## Animación de poblaciones completas

Avanza por generaciones enteras. Actualiza histograma, mapa de calor y estadísticas, con inicio, pausa, reinicio y navegación.

In [ ]:
indice=0; pausada=False; salida_anim=widgets.Output()
def mostrar_generacion():
    s=historial[indice]
    with salida_anim:
        clear_output(wait=True); fig,(a,b)=plt.subplots(1,2,figsize=(13,4.5),gridspec_kw={'width_ratios':[1,1.5]})
        im=graficar_poblacion(a,b,s['poblacion'],f"Generación {s['generacion']}"); fig.colorbar(im,ax=b,label='Valor del gen')
        fig.suptitle(f"Mejor {s['mejor_fitness']} · promedio {s['fitness_promedio']:.2f} · peor {s['peor_fitness']} · diversidad {s['diversidad']:.3f}"); plt.tight_layout(); plt.show()
def paso(delta):
    global indice
    indice=max(0,min(len(historial)-1,indice+delta)); mostrar_generacion()
def iniciar(_):
    global indice,pausada
    pausada=False
    while indice<len(historial)-1 and not pausada:
        indice+=1; mostrar_generacion(); time.sleep(.12)
def pausar(_):
    global pausada
    pausada=True
def reiniciar(_):
    global indice,pausada
    pausada=True; indice=0; mostrar_generacion()
botones=[widgets.Button(description=x) for x in ('Iniciar','Pausar','Reiniciar','◀ Anterior','Siguiente ▶')]
botones[0].on_click(iniciar); botones[1].on_click(pausar); botones[2].on_click(reiniciar)
botones[3].on_click(lambda _:paso(-1)); botones[4].on_click(lambda _:paso(1))
display(widgets.HBox(botones),salida_anim); mostrar_generacion()

La pausa es cooperativa y conserva la generación visible. Los controles paso adelante y atrás recorren las instantáneas.

## Experimentos con semillas y parámetros

Comparamos fitness final, generación de convergencia, promedio y diversidad final, y tiempo. Si fitness cero no aparece, convergencia se informa como “No” y se conserva el mejor resultado hallado.

In [ ]:
def comparar_corridas(semillas,mutacion,torneo,elitismo):
    filas=[]
    for semilla in semillas:
        cfg=ConfiguracionAG(80,200,mutacion,elitismo,torneo,semilla)
        t=time.perf_counter(); h=ejecutar_con_historial(cfg); dur=time.perf_counter()-t; fin=h[-1]
        conv=next((s['generacion'] for s in h if s['mejor_fitness']==0),None)
        filas.append({'semilla':semilla,'fitness_final':fin['mejor_fitness'],'generación_convergencia':conv if conv is not None else 'No',
                      'promedio_final':fin['fitness_promedio'],'diversidad_final':fin['diversidad'],'tiempo_s':dur})
    return pd.DataFrame(filas)
display(comparar_corridas([11,22,33],.15,3,2).round(3))